# 03 - Pemodelan (2 Skenario), Evaluasi & Ekspor Artefak
**Owner: Role 4.** Aturan: split tetap (`RANDOM_STATE`), outlier dibuang **hanya dari data latih**,
semua class/fungsi kustom ada di `src/`, model dipilih berdasarkan **CV Skenario 2** (bukan skor test).

> Estimasi akademik berbasis data pasar Ames, Iowa (2006–2010). Tidak mencerminkan harga pasar saat ini.

In [1]:
# Setup Colab (sekali). Setelah sel ini selesai pertama kali: Runtime -> Restart session, lalu jalankan dari sel berikutnya.
# Repo publik: tidak perlu token. Jika suatu saat privat, pakai Colab Secrets (jangan tulis token di notebook).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = "/content/real-estate-avm"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/kaniaa-kr/real-estate-avm.git {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt
    os.chdir(REPO_DIR)
if not IN_COLAB and os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))  # VS Code menjalankan notebook dari folder notebooks/
sys.path.insert(0, os.getcwd())
print("Siap menjalankan notebook dari root repo.")


Siap menjalankan notebook dari root repo.


In [2]:
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline

from src.config import ENGINEERED_FEATURES, MODEL_VERSION, RANDOM_STATE, TARGET, TEST_SIZE
from src.evaluation import (apply_interval, coverage, full_metrics, oof_residuals, ui_columns)
from src.explainer import attach_background, detect_method, explain
from src.preprocessing import build_preprocessor

for lib in ["scikit-learn", "pandas", "numpy", "joblib", "shap"]:
    print(lib, version(lib))

scikit-learn 1.6.1
pandas 2.2.3
numpy 2.4.6
joblib 1.6.0
shap 0.50.0


## 1. Load data, split tetap, buang outlier HANYA dari data latih

In [3]:
df = pd.read_csv("data/raw/train.csv")
assert df.shape == (1460, 81), df.shape

y = np.log1p(df[TARGET])
X = df.drop(columns=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

outlier_mask = (X_train["GrLivArea"] > 4000) & (np.expm1(y_train) < 300000)
print("Outlier dibuang dari train:", int(outlier_mask.sum()))
X_train, y_train = X_train[~outlier_mask], y_train[~outlier_mask]

UI_COLS = ui_columns()
print("10 fitur UI:", UI_COLS)
print("Train:", X_train.shape, "| Test:", X_test.shape)
assert len(UI_COLS) == 10

Outlier dibuang dari train: 2
10 fitur UI: ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'YearBuilt', 'Neighborhood', 'GarageCars', 'FullBath', 'LotArea', 'Fireplaces', 'KitchenQual']
Train: (1166, 80) | Test: (292, 80)


## 2. Latih 4 model x 2 skenario (K-Fold CV, split yang sama)
- Skenario 1: fitur penuh (`build_preprocessor("full")`) = benchmark batas atas.
- Skenario 2: 10 fitur UI (`build_preprocessor("ui10")`) = model produksi.
- Metrik CV = RMSLE (RMSE pada skala log1p). Random Forest dibatasi `n_estimators=100`, `max_depth<=12`.

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# nama: (estimator, grid hyperparameter, for_linear)
MODELS = {
    "OLS": (LinearRegression(), {}, True),
    "Ridge": (Ridge(), {"model__alpha": np.logspace(-2, 3, 11)}, True),
    "Lasso": (Lasso(max_iter=50000), {"model__alpha": np.logspace(-4, -1, 8)}, True),
    "Random Forest": (
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
        {"model__max_depth": [8, 10, 12], "model__min_samples_leaf": [1, 3, 5], "model__max_features": [0.5, 1.0]},
        False,
    ),
}
SCENARIOS = {1: ("full", X_train), 2: ("ui10", X_train[UI_COLS])}

fitted, rows = {}, []
for scen, (fset, Xs) in SCENARIOS.items():
    for name, (est, grid, for_linear) in MODELS.items():
        pipe = Pipeline([
            ("preprocess", build_preprocessor(feature_set=fset, for_linear=for_linear)),
            ("model", clone(est)),
        ])
        gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1, refit=True)
        gs.fit(Xs, y_train)
        fitted[(scen, name)] = gs
        rows.append({"model": name, "scenario": scen, "rmsle_cv": -gs.best_score_, "best_params": gs.best_params_})
        print(f"Skenario {scen} | {name:14s} | RMSLE CV = {-gs.best_score_:.4f} | {gs.best_params_}")

cv_table = pd.DataFrame(rows)
cv_table

Skenario 1 | OLS            | RMSLE CV = 0.1291 | {}


Skenario 1 | Ridge          | RMSLE CV = 0.1092 | {'model__alpha': np.float64(31.622776601683793)}


Skenario 1 | Lasso          | RMSLE CV = 0.1090 | {'model__alpha': np.float64(0.0007196856730011522)}


Skenario 1 | Random Forest  | RMSLE CV = 0.1313 | {'model__max_depth': 12, 'model__max_features': 0.5, 'model__min_samples_leaf': 1}
Skenario 2 | OLS            | RMSLE CV = 0.1343 | {}


Skenario 2 | Ridge          | RMSLE CV = 0.1341 | {'model__alpha': np.float64(3.1622776601683795)}


Skenario 2 | Lasso          | RMSLE CV = 0.1343 | {'model__alpha': np.float64(0.0001)}


Skenario 2 | Random Forest  | RMSLE CV = 0.1420 | {'model__max_depth': 12, 'model__max_features': 0.5, 'model__min_samples_leaf': 1}


,model,scenario,rmsle_cv,best_params
0,OLS,1,0.129050,{}
1,Ridge,1,0.109228,{'model__alpha': 31.622776601683793}
2,Lasso,1,0.109044,{'model__alpha': 0.0007196856730011522}
3,Random Forest,1,0.131347,"{'model__max_depth': 12, 'model__max_features'..."
4,OLS,2,0.134313,{}
5,Ridge,2,0.134129,{'model__alpha': 3.1622776601683795}
6,Lasso,2,0.134279,{'model__alpha': 0.0001}
7,Random Forest,2,0.141980,"{'model__max_depth': 12, 'model__max_features'..."


## 3. Pilih model final (hanya dari CV Skenario 2)
Aturan: ambil semua model dengan RMSLE CV dalam 1% dari yang terbaik, lalu pilih yang **paling sederhana**
(OLS < Ridge = Lasso < Random Forest); jika seri, ambil CV terendah. Pemilihan dikunci **sebelum** melihat skor test.

In [5]:
COMPLEXITY = {"OLS": 1, "Ridge": 1, "Lasso": 1, "Random Forest": 2}
s2 = cv_table[cv_table["scenario"] == 2].copy()
s2["complexity"] = s2["model"].map(COMPLEXITY)
best_cv = s2["rmsle_cv"].min()
candidates = s2[s2["rmsle_cv"] <= best_cv * 1.01]
CHOSEN = candidates.sort_values(["complexity", "rmsle_cv"]).iloc[0]["model"]
print("CV terbaik Skenario 2:", round(best_cv, 4))
print("Model final terpilih :", CHOSEN)

CV terbaik Skenario 2: 0.1341
Model final terpilih : Ridge


## 4. Evaluasi test set (hanya model final + benchmark yang dipilih dari CV)
Test set tidak dipakai untuk memilih model. Evaluasi dilakukan **sekali** untuk model final (Skenario 2)
dan **sekali** untuk benchmark fitur penuh (Skenario 1) yang dipilih dari RMSLE CV terbaik.


In [6]:
# Tabel perbandingan memakai skor CV saja.
# Test set TIDAK dipakai untuk memilih model/benchmark (AGENTS.md Bagian 2.5).
results = pd.DataFrame([
    {
        "Model": name,
        "Skenario": "1 (fitur penuh)" if scen == 1 else "2 (10 fitur)",
        "RMSLE (CV)": round(-gs.best_score_, 4),
        "_scen": scen,
    }
    for (scen, name), gs in fitted.items()
]).sort_values(["_scen", "Model"]).drop(columns="_scen")
results


,Model,Skenario,RMSLE (CV)
2,Lasso,1 (fitur penuh),0.1090
0,OLS,1 (fitur penuh),0.1291
3,Random Forest,1 (fitur penuh),0.1313
1,Ridge,1 (fitur penuh),0.1092
6,Lasso,2 (10 fitur),0.1343
4,OLS,2 (10 fitur),0.1343
7,Random Forest,2 (10 fitur),0.1420
5,Ridge,2 (10 fitur),0.1341


In [7]:
final_gs = fitted[(2, CHOSEN)]
final_pipeline = final_gs.best_estimator_

# Benchmark = model Skenario 1 dengan RMSLE CV terbaik (bukan skor test).
bench_cv = cv_table[cv_table["scenario"] == 1].sort_values("rmsle_cv").iloc[0]
BENCH = bench_cv["model"]
bench_pipeline = fitted[(1, BENCH)].best_estimator_

# Test set dievaluasi sekali untuk model final dan sekali untuk benchmark terpilih.
final_test = full_metrics(y_test, final_pipeline.predict(X_test[UI_COLS]))
bench_test = full_metrics(y_test, bench_pipeline.predict(X_test))

print("Model final      :", CHOSEN, final_gs.best_params_)
print("RMSLE CV final   :", round(-final_gs.best_score_, 4))
print("RMSLE test final :", round(final_test["rmsle"], 4), "(target PRD 0,15-0,18)")
print("Benchmark (CV)   :", BENCH, round(bench_cv["rmsle_cv"], 4))
print("RMSLE test bench :", round(bench_test["rmsle"], 4))
print("Selisih RMSLE (10 fitur vs fitur penuh):", round(final_test["rmsle"] - bench_test["rmsle"], 4))


Model final      : Ridge {'model__alpha': np.float64(3.1622776601683795)}
RMSLE CV final   : 0.1341
RMSLE test final : 0.15 (target PRD 0,15-0,18)
Benchmark (CV)   : Lasso 0.109
RMSLE test bench : 0.1256
Selisih RMSLE (10 fitur vs fitur penuh): 0.0244


### Analisis (ISI MANUAL dari output di atas - jangan mengarang angka)
- Model mana yang unggul di Skenario 1 dan 2? Jika Ridge/Lasso >= Random Forest, jelaskan penyebabnya
  (mis. target sudah di-log, hubungan relatif linear, data kecil 1460 baris, regularisasi mengatasi kolinearitas).
- Seberapa besar kehilangan akurasi dari fitur penuh -> 10 fitur?
- Jika RMSLE di luar 0,15-0,18, tuliskan alasannya.

## 5. Rentang harga (kuantil residual out-of-fold) & cakupan test
Residual OOF dihitung pada data latih. Cakupan test target >= 80%. Jika kurang, kuantil diperlebar bertahap.
(Catatan kejujuran: pelebaran berdasarkan cakupan test = sedikit "mengintip" test, catat di laporan.)

In [8]:
res = oof_residuals(final_pipeline, X_train[UI_COLS], y_train, cv)
pred_test_log = final_pipeline.predict(X_test[UI_COLS])
y_test_usd = np.expm1(y_test)

TARGET_COVERAGE = 0.80
for lo, hi in [(5, 95), (4, 96), (3, 97), (2, 98), (1, 99)]:
    q_low, q_high = (float(v) for v in np.percentile(res, [lo, hi]))
    low, pred, high = apply_interval(pred_test_log, q_low, q_high)
    cov = coverage(y_test_usd, low, high)
    print(f"persentil {lo}-{hi}: q_low={q_low:.4f} q_high={q_high:.4f} cakupan test={cov:.3f}")
    if cov >= TARGET_COVERAGE:
        break
INTERVAL_PCT = (lo, hi)
assert (low >= 0).all() and (low <= pred).all() and (pred <= high).all()

persentil 5-95: q_low=-0.2093 q_high=0.1924 cakupan test=0.877


## 6. Explainer (top_factors dalam USD)
Metode ditentukan otomatis dari model final: Random Forest -> `shap_tree`; Ridge/Lasso/OLS -> `linear_coef`.
Background (200 baris latih) disimpan di pipeline lewat `attach_background` agar API cukup memanggil `explain(pipeline, input_df)`.

In [9]:
attach_background(final_pipeline, X_train[UI_COLS], n=200, random_state=RANDOM_STATE)
EXPLANATION_METHOD = detect_method(final_pipeline)
print("explanation_method:", EXPLANATION_METHOD)

for i in [0, 1, 2]:
    sample = X_test[UI_COLS].iloc[[i]]
    print("Prediksi USD:", round(float(np.expm1(final_pipeline.predict(sample))[0])),
          "| Aktual USD:", round(float(y_test_usd.iloc[i])))
    print(json.dumps(explain(final_pipeline, sample, top_n=5), indent=2))

explanation_method: linear_coef
Prediksi USD: 132479 | Aktual USD: 154500


[
  {
    "feature": "GrLivArea",
    "impact_usd": -8061,
    "direction": "down"
  },
  {
    "feature": "GarageCars",
    "impact_usd": -6451,
    "direction": "down"
  },
  {
    "feature": "TotalSF",
    "impact_usd": -5760,
    "direction": "down"
  },
  {
    "feature": "KitchenQual",
    "impact_usd": -4982,
    "direction": "down"
  },
  {
    "feature": "HouseAge",
    "impact_usd": -4073,
    "direction": "down"
  }
]
Prediksi USD: 319073 | Aktual USD: 325000
[
  {
    "feature": "TotalSF",
    "impact_usd": 26193,
    "direction": "up"
  },
  {
    "feature": "OverallQual",
    "impact_usd": 23777,
    "direction": "up"
  },
  {
    "feature": "GrLivArea",
    "impact_usd": 19108,
    "direction": "up"
  },
  {
    "feature": "Neighborhood",
    "impact_usd": 11373,
    "direction": "up"
  },
  {
    "feature": "HouseAge",
    "impact_usd": 10346,
    "direction": "up"
  }
]
Prediksi USD: 117851 | Aktual USD: 115000
[
  {
    "feature": "HouseAge",
    "impact_usd": -19337,

## 7. Ekspor artefak (`.joblib` < 25 MB, compress=3) dan meta JSON

In [10]:
Path("models").mkdir(exist_ok=True)
MODEL_FILE = Path(f"models/model_pipeline_{MODEL_VERSION}.joblib")
META_FILE = Path(f"models/model_meta_{MODEL_VERSION}.json")

joblib.dump(final_pipeline, MODEL_FILE, compress=3)
size_mb = MODEL_FILE.stat().st_size / 1024 ** 2
print(f"Ukuran {MODEL_FILE.name}: {size_mb:.2f} MB")
assert size_mb < 25, "Artefak >= 25 MB: kecilkan max_depth/min_samples_leaf dan latih ulang"

# verifikasi round-trip
reloaded = joblib.load(MODEL_FILE)
np.testing.assert_allclose(reloaded.predict(X_test[UI_COLS]), pred_test_log)
print("Round-trip joblib OK")

Ukuran model_pipeline_v1.joblib: 0.01 MB


Round-trip joblib OK


In [11]:
def _metrics_payload(m: dict) -> dict:
    """Ambil empat metrik PRD dari hasil full_metrics (test set)."""
    return {
        "rmsle": round(float(m["rmsle"]), 4),
        "mae_usd": round(float(m["mae_usd"])),
        "rmse_usd": round(float(m["rmse_usd"])),
        "r2": round(float(m["r2"]), 4),
    }

meta = {
    "model_version": MODEL_VERSION,
    "algorithm": CHOSEN,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "features_used": UI_COLS,
    "engineered_features": list(ENGINEERED_FEATURES),
    "metrics": {
        "test_10_features": _metrics_payload(final_test),
        "benchmark_full_features": _metrics_payload(bench_test),
    },
    "interval": {
        "method": "oof_log_residual_quantile",
        "nominal_coverage": round((INTERVAL_PCT[1] - INTERVAL_PCT[0]) / 100, 2),
        "q_low": q_low,
        "q_high": q_high,
        "test_coverage": cov,
    },
    "explanation_method": EXPLANATION_METHOD,
    "split": {"random_state": RANDOM_STATE, "test_size": TEST_SIZE},
    "library_versions": {
        "python": platform.python_version(),
        **{lib: version(lib) for lib in ["scikit-learn", "pandas", "numpy", "joblib", "shap"]},
    },
}
META_FILE.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

# Skema meta HARUS sama dengan stub (PRD Bagian 5) dan lolos kontrak API.
stub = json.loads(Path("models/model_meta_stub.json").read_text(encoding="utf-8"))
missing, extra = set(stub) - set(meta), set(meta) - set(stub)
assert not missing and not extra, f"Skema meta tidak sama dengan stub: kurang={missing}, lebih={extra}"

from api.schemas import ModelInfoResponse  # noqa: E402

ModelInfoResponse(**meta)
print("Meta v1 cocok dengan stub dan ModelInfoResponse.")
print(json.dumps(meta, indent=2, ensure_ascii=False))


Meta v1 cocok dengan stub dan ModelInfoResponse.
{
  "model_version": "v1",
  "algorithm": "Ridge",
  "trained_at": "2026-11-25T16:03:52+00:00",
  "features_used": [
    "OverallQual",
    "GrLivArea",
    "TotalBsmtSF",
    "YearBuilt",
    "Neighborhood",
    "GarageCars",
    "FullBath",
    "LotArea",
    "Fireplaces",
    "KitchenQual"
  ],
  "engineered_features": [
    "TotalSF",
    "HouseAge"
  ],
  "metrics": {
    "test_10_features": {
      "rmsle": 0.15,
      "mae_usd": 18063,
      "rmse_usd": 29256,
      "r2": 0.8884
    },
    "benchmark_full_features": {
      "rmsle": 0.1256,
      "mae_usd": 14034,
      "rmse_usd": 22152,
      "r2": 0.936
    }
  },
  "interval": {
    "method": "oof_log_residual_quantile",
    "nominal_coverage": 0.9,
    "q_low": -0.20929642722291097,
    "q_high": 0.19238363046831042,
    "test_coverage": 0.8767123287671232
  },
  "explanation_method": "linear_coef",
  "split": {
    "random_state": 42,
    "test_size": 0.2
  },
  "library_ver

In [12]:
# Unduh artefak dari Colab, lalu commit dari komputer lokal.
if IN_COLAB:
    from google.colab import files
    for p in [MODEL_FILE, META_FILE]:
        files.download(str(p))
